# Bronze ingestion and source audit

## Objective
Validate the five M5 source files, ingest them as typed Delta tables, and preserve enough lineage to reproduce every downstream dataset. Bronze deliberately retains the original wide sales layout: reshaping and business joins belong to Silver, while Bronze remains directly auditable against each CSV.

No imputation, normalization, aggregation, filtering, or feature engineering occurs here. The only added columns are source filename, SHA-256 checksum, UTC ingestion timestamp, and pipeline run ID.


In [ ]:
profile = "dev"
run_id = "notebook-bronze"
force = False
execute_stage = False
seed = 42


## Ingestion decisions

- Sales identifiers are strings and `d_*` demand columns are integers.
- Weekly selling price is double precision; store, item, and week form its business key.
- Calendar and submission schemas are inferred because their mixed fields are validated by required-column contracts first.
- CSV parsing uses `FAILFAST`; malformed records stop the stage instead of being silently dropped.
- Delta writes replace the table atomically, making reruns deterministic for a fixed source checksum.


In [ ]:
from IPython.display import display
import pandas as pd

from retail_forecasting.config import load_config
from retail_forecasting.data.bronze import run_bronze
from retail_forecasting.data.quality import validate_source_files
from retail_forecasting.data.spark import get_spark, table_path

config = load_config(profile)
source_report = validate_source_files(config)
source_audit = pd.DataFrame(source_report["checks"])
source_audit["size_mib"] = source_audit["size_bytes"] / 1024**2
display(source_audit[["file", "size_mib", "columns", "missing_required_columns", "valid"]])
assert source_report["valid"], source_report


In [ ]:
execute_stage_enabled = str(execute_stage).strip().lower() in {"1", "true", "yes"}
stage_result = run_bronze(config, run_id) if execute_stage_enabled else {"status": "reusing existing Bronze tables"}
stage_result


## Delta table audit
The audit below verifies row and column counts and confirms that each Delta table contains exactly one source file and checksum. Checksums are displayed only as short prefixes for readability; the complete value remains stored on every Bronze row.


In [ ]:
spark = get_spark(config, "notebook-bronze-audit")
table_rows = []
for filename in config.data.required_files:
    table_name = filename.removesuffix(".csv")
    frame = spark.read.format("delta").load(str(table_path(config, "bronze", table_name)))
    lineage = frame.select("_source_file", "_source_sha256", "_run_id").distinct().collect()
    table_rows.append({
        "table": table_name,
        "rows": frame.count(),
        "columns": len(frame.columns),
        "lineage_records": len(lineage),
        "source_file": lineage[0]["_source_file"] if lineage else None,
        "sha256_prefix": lineage[0]["_source_sha256"][:12] if lineage else None,
        "run_id": lineage[0]["_run_id"] if lineage else None,
    })
bronze_audit = pd.DataFrame(table_rows)
display(bronze_audit)
assert bronze_audit["lineage_records"].eq(1).all()


## Raw sales structure
The evaluation sales file has one row per original M5 series and one column per day. This inspection makes the dimensionality explicit before Silver expands it to the daily panel.


In [ ]:
sales = spark.read.format("delta").load(str(table_path(config, "bronze", "sales_train_evaluation")))
day_columns = sorted(
    [column for column in sales.columns if column.startswith("d_")],
    key=lambda value: int(value.removeprefix("d_")),
)
structure = pd.DataFrame([{
    "series_rows": sales.count(),
    "identifier_columns": 6,
    "daily_columns": len(day_columns),
    "first_day": day_columns[0],
    "last_day": day_columns[-1],
}])
display(structure)
display(sales.select("id", "item_id", "dept_id", "cat_id", "store_id", "state_id", day_columns[0], day_columns[-1]).limit(10).toPandas())


## What Bronze does not claim
Bronze validation proves file presence, minimum schema, parseability, and lineage. It does not prove business-key uniqueness, calendar completeness, non-negative demand, or join coverage; those checks require normalized data and are performed in Silver. Source data, Delta files, and checksums remain outside Git.


In [ ]:
spark.stop()
